# Como-Article Ground/Vehicle MVP

Spec: `docs/specs/0006-como-article-ground-vehicle-mvp.md`

This is the canonical full-shard workflow for the current project. It uses a Spark-native prefilter before the Python extraction UDF, then writes the gold candidate dataset and compact analysis CSVs.

```text
Bronze corpus segments
  -> native Spark rlike prefilter for curated GROUND como article
  -> Python UDF extraction only on plausible rows
  -> gold Parquet and WebApp-ready CSV outputs
```


In [1]:
import os
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src/tal_qual").exists():
            return candidate
        if (candidate / "work/src/tal_qual").exists():
            return candidate / "work"
    raise RuntimeError(f"Could not find repository root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
SRC_DIR = REPO_ROOT / "src"
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

REPO_ROOT


PosixPath('/home/jovyan/work')

In [2]:
import os
import tempfile
import zipfile

from pyspark.sql import SparkSession

SPARK_MASTER = os.environ.get("TAL_QUAL_SPARK_MASTER", "local[4]")
SPARK_PARALLELISM = os.environ.get("TAL_QUAL_SPARK_PARALLELISM", "4")
SPARK_SHUFFLE_PARTITIONS = os.environ.get("TAL_QUAL_SPARK_SHUFFLE_PARTITIONS", "4")

spark = (
    SparkSession.builder
    .master(SPARK_MASTER)
    .appName("tal-qual-como-article-ground-vehicle-optimized-mvp")
    .config("spark.default.parallelism", SPARK_PARALLELISM)
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS)
    .config("spark.sql.files.maxPartitionBytes", str(256 * 1024 * 1024))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

package_zip = Path(tempfile.gettempdir()) / "tal_qual_src.zip"
with zipfile.ZipFile(package_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted((SRC_DIR / "tal_qual").rglob("*.py")):
        archive.write(file_path, file_path.relative_to(SRC_DIR))

spark.sparkContext.addPyFile(str(package_zip))
SPARK_MASTER, spark.version


('local[4]', '4.1.1')

In [3]:
from tal_qual.bronze import BRONZE_OUTPUT_PATH, RAW_CORPUS_INPUT, prepare_bronze_dataframe, write_bronze_parquet
from tal_qual.como_article_ground_vehicle import (
    COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH,
    COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    como_article_examples_dataframe,
    como_article_ground_counts_dataframe,
    como_article_ground_vehicle_counts_dataframe,
    como_article_review_sample_dataframe,
    como_article_vehicle_counts_dataframe,
    como_article_vehicle_ground_counts_dataframe,
    prefilter_como_article_ground_vehicle_bronze_dataframe,
    prepare_como_article_ground_vehicle_dataframe,
    write_como_article_csv_outputs,
    write_como_article_ground_vehicle_parquet,
)


## Load Or Build Bronze

The extractor expects the bronze schema from notebook 02. If bronze output is absent, this notebook builds it from the configured raw shard.


In [4]:
bronze_path = REPO_ROOT / BRONZE_OUTPUT_PATH
raw_path = REPO_ROOT / RAW_CORPUS_INPUT

if bronze_path.exists():
    bronze_df = spark.read.parquet(str(bronze_path))
else:
    bronze_df = prepare_bronze_dataframe(spark, raw_path)
    write_bronze_parquet(bronze_df, bronze_path)

bronze_df.printSchema()


root
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- text_original: string (nullable = true)
 |-- text_normalized: string (nullable = true)
 |-- match_text: string (nullable = true)



## Prefilter And Extract Spec-0006 Candidates

The native prefilter is intentionally broad but cheap. It reduces the number of rows sent through the Python UDF.


In [5]:
prefiltered_bronze_df = prefilter_como_article_ground_vehicle_bronze_dataframe(bronze_df).persist()
prefiltered_count = prefiltered_bronze_df.count()
print(f"Prefiltered bronze rows: {prefiltered_count:,}")

candidates_df = prepare_como_article_ground_vehicle_dataframe(prefiltered_bronze_df).persist()
candidate_count = candidates_df.count()
print(f"Spec-0006 candidates: {candidate_count:,}")
candidates_df


Prefiltered bronze rows: 333


/usr/local/spark/python/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Spec-0006 candidates: 204


DataFrame[candidate_id: string, source_file: string, original_line_id: bigint, segment_id: int, pattern_type: string, connector: string, connector_text: string, matched_text: string, candidate_full_text: string, text_before: string, tenor_text: string, tenor_lemma: string, tenor_confidence: string, ground_text: string, ground_lemma: string, ground_type: string, ground_source: string, vehicle_text: string, vehicle_lemma: string, vehicle_head: string, vehicle_head_lemma: string, vehicle_phrase_length_tokens: int, filter_label: string, reject_reason: string, confidence: double, needs_review: boolean, char_start: int, char_end: int, connector_start: int, connector_end: int, vehicle_start: int, vehicle_end: int]

In [6]:
candidates_df.printSchema()


root
 |-- candidate_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- original_line_id: long (nullable = true)
 |-- segment_id: integer (nullable = true)
 |-- pattern_type: string (nullable = true)
 |-- connector: string (nullable = true)
 |-- connector_text: string (nullable = true)
 |-- matched_text: string (nullable = true)
 |-- candidate_full_text: string (nullable = true)
 |-- text_before: string (nullable = true)
 |-- tenor_text: string (nullable = true)
 |-- tenor_lemma: string (nullable = true)
 |-- tenor_confidence: string (nullable = true)
 |-- ground_text: string (nullable = true)
 |-- ground_lemma: string (nullable = true)
 |-- ground_type: string (nullable = true)
 |-- ground_source: string (nullable = true)
 |-- vehicle_text: string (nullable = true)
 |-- vehicle_lemma: string (nullable = true)
 |-- vehicle_head: string (nullable = true)
 |-- vehicle_head_lemma: string (nullable = true)
 |-- vehicle_phrase_length_tokens: integer (nullable = true

## Write Gold Dataset And Visualization CSVs

In [7]:
write_como_article_ground_vehicle_parquet(candidates_df, REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_OUTPUT_PATH)
write_como_article_csv_outputs(
    candidates_df,
    ground_vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_VEHICLE_COUNTS_OUTPUT_PATH,
    vehicle_ground_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_GROUND_COUNTS_OUTPUT_PATH,
    ground_counts_path=REPO_ROOT / COMO_ARTICLE_GROUND_COUNTS_OUTPUT_PATH,
    vehicle_counts_path=REPO_ROOT / COMO_ARTICLE_VEHICLE_COUNTS_OUTPUT_PATH,
    examples_path=REPO_ROOT / COMO_ARTICLE_EXAMPLES_OUTPUT_PATH,
    review_sample_path=REPO_ROOT / COMO_ARTICLE_REVIEW_SAMPLE_OUTPUT_PATH,
)


## Inspect Visualization Tables

These displays use small materialized pandas tables for readability. The durable CSV outputs above remain the source of truth for downstream visualization work.


In [8]:
from pyspark.sql.functions import col

def display_table(spark_df, limit=30):
    display(spark_df.limit(limit).toPandas())


### Ground -> Vehicle Pairs


In [9]:
display_table(
    como_article_ground_vehicle_counts_dataframe(candidates_df).select(
        col("ground_lemma").alias("ground"),
        col("vehicle_head_lemma").alias("vehicle_head"),
        "count",
    ),
    limit=40,
)


,ground,vehicle_head,count
0,cair,luva,49
1,cair,bomba,24
2,forte,touro,4
3,assentar,luva,3
4,caber,luva,3
5,encaixar,luva,3
6,flutuar,borboleta,3
7,rápido,flecha,3
8,rápido,raio,3
9,cair,verdadeiro,2


### Vehicle -> Ground Pairs


In [10]:
display_table(
    como_article_vehicle_ground_counts_dataframe(candidates_df).select(
        col("vehicle_head_lemma").alias("vehicle_head"),
        col("ground_lemma").alias("ground"),
        "count",
    ),
    limit=40,
)


,vehicle_head,ground,count
0,luva,cair,49
1,bomba,cair,24
2,touro,forte,4
3,borboleta,flutuar,3
4,flecha,rápido,3
5,luva,assentar,3
6,luva,caber,3
7,luva,encaixar,3
8,raio,rápido,3
9,avião,voar,2


### Ground Counts


In [11]:
display_table(como_article_ground_counts_dataframe(candidates_df), limit=40)


,ground_lemma,count
0,cair,83
1,trabalhar,15
2,forte,12
3,rápido,9
4,correr,6
5,duro,6
6,grande,6
7,voar,6
8,alto,5
9,encaixar,5


### Vehicle Counts


In [12]:
display_table(como_article_vehicle_counts_dataframe(candidates_df), limit=40)


,vehicle_head_lemma,count
0,luva,58
1,bomba,25
2,raio,4
3,touro,4
4,verdadeiro,4
5,bala,3
6,borboleta,3
7,flecha,3
8,avião,2
9,caixa,2


### Examples


In [13]:
display_table(
    como_article_examples_dataframe(candidates_df, limit=100).select(
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "candidate_full_text",
    ),
    limit=50,
)


,ground_text,connector_text,vehicle_text,tenor_text,candidate_full_text
0,alegre,como um,passarinho,olhos e saltou da cama,"Ela abriu os olhos e saltou da cama, alegre co..."
1,alto,como uma,caixa,vamos sentar sobre algo mais,Para iniciar com tranqüilidade vamos sentar so...
2,alta,como uma,estrela,meu baço é tão curto,"gro vê-la, cego ser e não vê-la, eu preferira:..."
3,alta,como um,pau de sebo,A criada,"A criada, alta como um pau de sebo"
4,alto,como um,pássaro com asas de anjo,céu além das estrelas Voarei,"Voarei além do céu, além das estrelas Voarei a..."
5,alto,como um,super heroi,já sonhou que estava voando,Se você já sonhou que estava voando alto como ...
6,assenta,como uma,luva,esquerda a quem o futuro,"mundo tem lutado nasce da ideia de que ""o bem""..."
7,assenta,como uma,luva,seguramente um daqueles a quem,Álvaro Magalhães é seguramente um daqueles a q...
8,assenta,como uma,luva no dia de hoje,ver aqui há um que,"NA LISTA de livros que se pode ver [aqui], há ..."
9,baixo,como um,barco,um sofá dinamarquês longo e,empo eu supri minha sala com uma moderna répli...


### Review Sample


In [14]:
display_table(
    como_article_review_sample_dataframe(candidates_df, limit=100).select(
        "ground_type",
        "ground_text",
        "connector_text",
        "vehicle_text",
        "tenor_text",
        "confidence",
        "needs_review",
        "candidate_full_text",
    ),
    limit=50,
)


,ground_type,ground_text,connector_text,vehicle_text,tenor_text,confidence,needs_review,candidate_full_text
0,salient_verb,Flutuar,como uma,borboleta,,0.75,True,Flutuar como uma borboleta
1,quality_adjective,Forte,como um,touro e alfabetizada,,0.75,True,Forte como um touro e alfabetizada
2,quality_adjective,Livre,como um,táxi,,0.75,True,Livre como um táxi
3,quality_adjective,Rápida,como uma,flecha,,0.75,True,Rápida como uma flecha
4,quality_adjective,Calmo,Como Uma,Bomba,,0.75,True,Calmo Como Uma Bomba
5,salient_verb,Correu,como uma,bala para a pensão,,0.75,True,Correu como uma bala para a pensão
6,quality_adjective,Cego,como um,morcego,,0.75,True,Cego como um morcego
7,quality_adjective,Brilhante,como uma,caixa de celofane âmbar,,0.75,True,Brilhante como uma caixa de celofane âmbar
8,quality_adjective,baixo,como um,prego saindo de alguma madeira,ele irá obter bateu para,0.90,False,Ouvi dizer que você não furar sua cabeça no Ja...
9,salient_verb,caiu,como um,castelinho de areia,casar e de repente tudo,0.90,False,"ada tava numa boa com ele, viajando , fazendo ..."
